# 模型构建与参数管理

本notebook介绍PyTorch中模型构建和参数管理的核心技术:
- 层和块(Module)的概念
- 自定义层和模型
- 参数访问和初始化
- 参数共享
- 模型的保存和加载

掌握这些技术,你就能构建任意复杂的神经网络!

## 第一部分: 层和块

### 1.1 什么是块(Block)?

**块**是PyTorch中的核心抽象概念:
- **单个层**: 如`nn.Linear`, `nn.Conv2d`
- **多个层的组合**: 如ResNet中的残差块
- **整个模型**: 完整的神经网络

**块的特点**:
1. 接受输入数据
2. 生成输出
3. 包含可学习的参数
4. 可以计算梯度(自动微分)

**块的组合**:
```
简单块 → 组合成复杂块 → 组合成更复杂的块 → 完整模型
```

### 1.2 使用Sequential构建模型

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

In [ ]:
# Sequential: 按顺序执行的容器
net = nn.Sequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

X = torch.rand(2, 20)
output = net(X)

print(f'输入形状: {X.shape}')
print(f'输出形状: {output.shape}')
print(f'\n模型结构:')
print(net)

**Sequential的工作原理**:
- 维护一个有序的Module列表
- 按顺序执行每个模块
- 每个模块的输出是下一个模块的输入
- `net(X)` 实际上调用 `net.__call__(X)`

### 1.3 自定义块

**为什么需要自定义块?**
- Sequential只能按顺序执行
- 有些模型需要更复杂的控制流
- 需要实现特殊的计算逻辑

**自定义块的要求**:
1. 继承`nn.Module`
2. 在`__init__`中定义层和参数
3. 实现`forward`方法定义前向传播

In [ ]:
# 自定义MLP块
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)
    
    def forward(self, X):
        # 自定义前向传播逻辑
        return self.out(F.relu(self.hidden(X)))

net_custom = MLP()
print('自定义MLP:')
print(net_custom)
print(f'\n输出形状: {net_custom(X).shape}')

### 1.4 在forward中使用控制流

In [ ]:
class FlexibleMLP(nn.Module):
    """带有控制流的MLP"""
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(20, 256)
        self.linear2 = nn.Linear(256, 256)
        self.linear3 = nn.Linear(256, 10)
    
    def forward(self, X):
        H = F.relu(self.linear1(X))
        
        # 控制流: 条件执行
        if H.abs().sum() > 1:
            H = F.relu(self.linear2(H))  # 额外的层
        else:
            H = H / 2  # 缩放
        
        return self.linear3(H)

flexible_net = FlexibleMLP()
print('灵活的MLP(带控制流):')
print(flexible_net(X).shape)

### 1.5 嵌套块

In [ ]:
# 定义一个块工厂函数
def block1():
    return nn.Sequential(
        nn.Linear(20, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU()
    )

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 嵌套块: 将block1的输出作为下一个block1的输入
        net.add_module(f'block{i}', block1())
    return net

# 构建嵌套网络
nested_net = nn.Sequential(block2(), nn.Linear(32, 10))

print('嵌套网络结构:')
print(nested_net)
print(f'\n输出形状: {nested_net(torch.rand(2, 20)).shape}')

---

## 第二部分: 参数管理

### 2.1 参数访问

**为什么需要访问参数?**
- 调试和诊断
- 可视化权重
- 迁移学习
- 参数剪枝

In [ ]:
# 创建一个简单的网络
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

X = torch.rand(2, 4)
print('前向传播输出:')
print(net(X))

**访问特定层的参数**

In [ ]:
# 访问第3个模块(第二个线性层)的参数
print('第二个线性层的参数:')
print(net[2].state_dict())
print(f'\n权重形状: {net[2].weight.shape}')
print(f'偏置形状: {net[2].bias.shape}')

**访问参数的值和梯度**

In [ ]:
# 参数类型
print(f'参数类型: {type(net[2].bias)}')

# 参数值
print(f'\n偏置参数: {net[2].bias}')
print(f'偏置值: {net[2].bias.data}')

# 梯度(训练前为None)
print(f'\n梯度是否为None: {net[2].weight.grad is None}')

In [ ]:
# 进行一次反向传播后查看梯度
net(X).sum().backward()
print('反向传播后的梯度:')
print(net[2].weight.grad)

### 2.2 一次性访问所有参数

In [ ]:
# 访问第一层的参数
print('第一层的参数:')
print(*[(name, param.shape) for name, param in net[0].named_parameters()])

# 访问所有层的参数
print('\n所有层的参数:')
print(*[(name, param.shape) for name, param in net.named_parameters()])

In [ ]:
# 通过state_dict访问
print('\n通过state_dict访问:')
print(net.state_dict().keys())
print(f'\n第二层偏置: {net.state_dict()["2.bias"].data}')

### 2.3 嵌套块的参数访问

In [ ]:
# 访问嵌套网络的参数
print('嵌套网络的所有参数名称:')
for name, param in nested_net.named_parameters():
    print(f'{name}: {param.shape}')

---

## 第三部分: 参数初始化

### 3.1 内置初始化方法

In [ ]:
# 创建一个新网络
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))

# 1. 正态分布初始化
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)

net.apply(init_normal)
print('正态分布初始化后的权重:')
print(net[0].weight.data[0])

In [ ]:
# 2. 常数初始化
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
print('常数初始化后的权重:')
print(net[0].weight.data[0])

In [ ]:
# 3. Xavier初始化(适合Sigmoid/Tanh)
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

net.apply(init_xavier)
print('Xavier初始化后的权重:')
print(net[0].weight.data[0])

In [ ]:
# 4. He初始化(适合ReLU)
def init_he(m):
    if type(m) == nn.Linear:
        nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')

net.apply(init_he)
print('He初始化后的权重:')
print(net[0].weight.data[0])

### 3.2 对不同层使用不同的初始化

In [ ]:
def init_custom(m):
    if type(m) == nn.Linear:
        print(f'初始化 {m}')
        nn.init.xavier_uniform_(m.weight)

# 只对第一层应用Xavier
net[0].apply(init_xavier)
# 对第三层应用He初始化
net[2].apply(init_he)

print('\n第一层权重:', net[0].weight.data[0][:3])
print('第二层权重:', net[2].weight.data[0][:3])

### 3.3 自定义初始化

In [ ]:
# 自定义初始化: 使用特殊的分布
def custom_init(m):
    if type(m) == nn.Linear:
        print(f'初始化 {m}')
        # 均匀分布 U(-10, 10), 但保留绝对值大于5的权重
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(custom_init)
print('\n自定义初始化后的权重:')
print(net[0].weight[:2])

### 3.4 直接设置参数

In [ ]:
# 直接修改参数
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42

print('直接设置后的权重:')
print(net[0].weight.data[0])

---

## 第四部分: 参数共享

**为什么需要参数共享?**
- 减少参数数量
- 提高模型效率
- 某些架构需要(如Siamese网络)

In [ ]:
# 创建一个共享层
shared_layer = nn.Linear(8, 8)

# 使用共享层多次
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    shared_layer,  # 第一次使用
    nn.ReLU(),
    shared_layer,  # 第二次使用(共享参数)
    nn.ReLU(),
    nn.Linear(8, 1)
)

print('网络结构(注意layer 2和4是同一个对象):')
print(net)
print(f'\n总参数数量: {sum(p.numel() for p in net.parameters())}')

In [ ]:
# 验证参数是否共享
print('layer[2]和layer[4]是否是同一个对象?', net[2] is net[4])
print('\nlayer[2]的权重:')
print(net[2].weight.data[0][:5])
print('\nlayer[4]的权重(应该相同):')
print(net[4].weight.data[0][:5])

In [ ]:
# 修改共享层的参数
net[2].weight.data[0, 0] = 100

print('修改layer[2]后:')
print('layer[2]的权重:', net[2].weight.data[0, 0])
print('layer[4]的权重(也改变了):', net[4].weight.data[0, 0])

---

## 第五部分: 自定义层

### 5.1 不带参数的自定义层

In [ ]:
class CenteredLayer(nn.Module):
    """将输入减去均值"""
    def __init__(self):
        super().__init__()
    
    def forward(self, X):
        return X - X.mean()

# 测试
layer = CenteredLayer()
test_input = torch.FloatTensor([1, 2, 3, 4, 5])
print('输入:', test_input)
print('输出(减去均值):', layer(test_input))
print('输出的均值(应该接近0):', layer(test_input).mean())

In [ ]:
# 将自定义层嵌入到网络中
net = nn.Sequential(
    nn.Linear(8, 128),
    CenteredLayer()
)

Y = net(torch.rand(4, 8))
print(f'输出的均值: {Y.mean():.6f}')

### 5.2 带参数的自定义层

In [ ]:
class MyLinear(nn.Module):
    """自定义全连接层"""
    def __init__(self, in_units, units):
        super().__init__()
        # 使用nn.Parameter定义参数
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))
    
    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

# 测试
linear = MyLinear(5, 3)
print('自定义线性层:')
print(f'权重形状: {linear.weight.shape}')
print(f'偏置形状: {linear.bias.shape}')

# 前向传播
test_input = torch.rand(2, 5)
output = linear(test_input)
print(f'\n输入形状: {test_input.shape}')
print(f'输出形状: {output.shape}')

In [ ]:
# 使用自定义层构建网络
net = nn.Sequential(
    MyLinear(64, 8),
    MyLinear(8, 1)
)

print('使用自定义层的网络:')
print(net)
print(f'\n输出形状: {net(torch.rand(2, 64)).shape}')

---

## 第六部分: 保存和加载模型

### 6.1 保存和加载张量

In [ ]:
import os

# 创建保存目录
os.makedirs('../data', exist_ok=True)

# 保存单个张量
x = torch.arange(4)
torch.save(x, '../data/x-tensor.pt')

# 加载
x2 = torch.load('../data/x-tensor.pt')
print('保存并加载的张量:')
print(x2)

In [ ]:
# 保存张量列表
y = torch.zeros(4)
torch.save([x, y], '../data/xy-tensors.pt')

x2, y2 = torch.load('../data/xy-tensors.pt')
print('加载的张量列表:')
print('x:', x2)
print('y:', y2)

In [ ]:
# 保存字典
mydict = {'x': x, 'y': y}
torch.save(mydict, '../data/mydict.pt')

mydict2 = torch.load('../data/mydict.pt')
print('加载的字典:')
print(mydict2)

### 6.2 保存和加载模型参数

In [ ]:
# 定义一个MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)
    
    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(2, 20)
Y = net(X)

print('原始模型输出:')
print(Y)

In [ ]:
# 保存模型参数
torch.save(net.state_dict(), '../data/mlp.params')
print('模型参数已保存到 ../data/mlp.params')

In [ ]:
# 加载模型参数
clone = MLP()
clone.load_state_dict(torch.load('../data/mlp.params'))
clone.eval()  # 设置为评估模式

Y_clone = clone(X)
print('克隆模型输出(应该与原始相同):')
print(Y_clone)
print(f'\n输出是否相同: {(Y == Y_clone).all()}')

**注意**: 
- `torch.save(net.state_dict())` 只保存参数,不保存模型结构
- 加载时需要先定义相同的模型结构
- 这是推荐的做法,因为模型代码可能包含任意Python代码

### 6.3 保存整个模型(不推荐)

In [ ]:
# 保存整个模型(包括结构)
torch.save(net, '../data/mlp-entire.pt')

# 加载
loaded_net = torch.load('../data/mlp-entire.pt')
print('加载的完整模型:')
print(loaded_net)
print(f'\n输出: {loaded_net(X)}')

---

## 小结

### 核心概念

1. **块(Module)**:
   - PyTorch的核心抽象
   - 可以是单层、多层组合或完整模型
   - 必须实现`__init__`和`forward`

2. **模型构建**:
   - `nn.Sequential`: 顺序执行
   - 自定义Module: 灵活的控制流
   - 嵌套块: 构建复杂架构

3. **参数管理**:
   - `state_dict()`: 获取所有参数
   - `named_parameters()`: 遍历参数
   - `apply()`: 批量应用函数

4. **参数初始化**:
   - Xavier: 适合Sigmoid/Tanh
   - He(Kaiming): 适合ReLU
   - 自定义初始化: 灵活控制

5. **参数共享**:
   - 使用同一个层对象
   - 减少参数量
   - 梯度会累积

6. **自定义层**:
   - 继承`nn.Module`
   - 使用`nn.Parameter`定义可学习参数
   - 实现`forward`方法

7. **模型保存**:
   - 推荐: `torch.save(model.state_dict())`
   - 加载: `model.load_state_dict(torch.load())`

### 最佳实践

```python
# 1. 定义模型
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 20)
        self.layer2 = nn.Linear(20, 10)
    
    def forward(self, x):
        return self.layer2(F.relu(self.layer1(x)))

# 2. 初始化
model = MyModel()
model.apply(lambda m: nn.init.xavier_uniform_(m.weight) if isinstance(m, nn.Linear) else None)

# 3. 训练
# ...

# 4. 保存
torch.save(model.state_dict(), 'model.pt')

# 5. 加载
model = MyModel()
model.load_state_dict(torch.load('model.pt'))
model.eval()
```

## 练习

1. **构建ResNet块**: 实现一个残差块(带跳跃连接)
2. **参数统计**: 编写函数统计模型的总参数量
3. **参数冻结**: 实现冻结某些层的参数(迁移学习)
4. **权重可视化**: 可视化第一层卷积核的权重
5. **自定义激活函数**: 实现Swish激活函数
6. **检查点**: 实现训练过程中定期保存模型
7. **模型比较**: 比较两个模型的参数是否相同
8. **参数裁剪**: 实现权重裁剪(将小于阈值的权重置零)